# 33. 합성 평가셋 — 생성 결과 수집과 검증

사전 등록: `docs/plans/NLR_EVALSET_PREREGISTRATION.md`
입력 생성: 노트북 32

사람이 GPT 로 만든 문장 파일을 프로젝트 안으로 들여와 **검증하고 해시로 고정한다.**
채점은 하지 않는다. 기준선 측정은 노트북 34 에서 한다.

## 0. 실행 조건과 한계

- **API 호출 0회.** 이미 만들어진 CSV 를 읽는다
- 원본 8개를 `evaluation_data/nlr_evalset/raw/` 에 **그대로** 복사한다. 정규화본은 따로 만든다
  (`data/survey/` 의 `raw` / `processed` 관례를 따른다)
- 한 번 복사한 뒤에는 `raw/` 를 읽으므로 다운로드 폴더가 없어도 재실행된다
- 분포 비교에서 **평가 데이터(golden set · 설문)를 읽기만** 한다. 수정하지 않는다
- 재생성 판정은 하지 않는다. 사전 등록 §6 이 사유를 형식 오류와 규칙 2·3 위반으로 한정했다

In [1]:
import hashlib
import json
import pathlib
import re
import shutil

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

# 사람이 GPT 로 만든 파일이 처음 놓인 곳. raw/ 로 옮긴 뒤에는 쓰이지 않는다.
SOURCE_DIR = pathlib.Path.home() / "Downloads"
SOURCE_GLOBS = {"A": "perfume_evalset_batch*_A.csv",
                "B": "32_evalset_batch*_generated_B.csv"}

N_SENTENCES = 3       # 사전 등록 §6
MAX_LEN = 100         # 길이 이상치 기준 (사전 등록 §6 은 '70자 내외')

print("REPORT_ONLY:", REPORT_ONLY, "/ 원본 위치:", SOURCE_DIR)

REPORT_ONLY: False / 원본 위치: C:\Users\SSAFY\Downloads


## 1. 경로 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
EVAL_DIR = PROJECT_ROOT / "evaluation_data" / "nlr_evalset"
RAW_DIR = EVAL_DIR / "raw"

INPUT_PATHS = {
    "answer_key": OUTPUT_DIR / "32_evalset_answer_key.csv",
    "manifest": OUTPUT_DIR / "32_evalset_manifest.json",
    "survey": PROJECT_ROOT / "data" / "survey" / "processed"
              / "survey_nlp_queries_candidates.csv",
    "golden": PROJECT_ROOT / "evaluation_data" / "stage1"
              / "13_stage1_golden_set_v1_200.xlsx",
    "perfumes_csv": PROJECT_ROOT / "perfumes.csv",
}
OUTPUT_PATHS = {
    "generated_a": EVAL_DIR / "generated_A.csv",
    "generated_b": EVAL_DIR / "generated_B.csv",
    "checksums": EVAL_DIR / "checksums.json",
    "validation": OUTPUT_DIR / "33_evalset_validation.csv",
    "distribution": OUTPUT_DIR / "33_evalset_distribution.csv",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {p.resolve() for p in INPUT_PATHS.values()} | {
    (PROJECT_ROOT / "perfumes.jsonl").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "16_golden_set_quality_audit_reviewed.csv").resolve(),
    (PROJECT_ROOT / "data" / "scent_knowledge" / "domain_lexicon_v1.csv").resolve(),
    (PROJECT_ROOT / "data" / "scent_knowledge" / "domain_lexicon_v1_1.csv").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 또는 raw/ 아래에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    allowed = path in ALLOWED_WRITES or RAW_DIR.resolve() in path.parents
    if not allowed:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    path.parent.mkdir(parents=True, exist_ok=True)
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
answer_key,d81e0f4fb2f1e1cd
manifest,c65a2719f19fbe00
survey,cb10e249489feb2e
golden,a56613ad4bb082d2
perfumes_csv,cec1ea0b49885303


## 2. 원본 수집 — `raw/` 에 그대로 복사한다

이미 `raw/` 에 있으면 다운로드 폴더를 보지 않는다. 재실행 가능하게 하기 위해서다.

In [3]:
collected = {}
for arm, pattern in SOURCE_GLOBS.items():
    in_raw = sorted(RAW_DIR.glob(pattern))
    if in_raw:
        paths = in_raw
        print(f"갈래 {arm}: raw/ 에서 {len(paths)}개")
    else:
        paths = sorted(SOURCE_DIR.glob(pattern))
        if not paths:
            raise FileNotFoundError(f"갈래 {arm} 원본을 찾지 못했습니다: {SOURCE_DIR / pattern}")
        print(f"갈래 {arm}: {SOURCE_DIR.name} 에서 {len(paths)}개 → raw/ 로 복사")
        copied = []
        for p in paths:
            dest = write_output(RAW_DIR / p.name, lambda d, s=p: shutil.copyfile(s, d))
            copied.append(dest or p)
        paths = copied
    collected[arm] = paths

raw_hashes = {p.name: sha256_file(p) for ps in collected.values() for p in ps}
display(pd.Series(raw_hashes, name="sha256").str.slice(0, 16).to_frame())

갈래 A: Downloads 에서 4개 → raw/ 로 복사
저장: evaluation_data\nlr_evalset\raw\perfume_evalset_batch1_A.csv
저장: evaluation_data\nlr_evalset\raw\perfume_evalset_batch2_A.csv
저장: evaluation_data\nlr_evalset\raw\perfume_evalset_batch3_A.csv
저장: evaluation_data\nlr_evalset\raw\perfume_evalset_batch4_A.csv
갈래 B: Downloads 에서 4개 → raw/ 로 복사
저장: evaluation_data\nlr_evalset\raw\32_evalset_batch1_generated_B.csv
저장: evaluation_data\nlr_evalset\raw\32_evalset_batch2_generated_B.csv
저장: evaluation_data\nlr_evalset\raw\32_evalset_batch3_generated_B.csv
저장: evaluation_data\nlr_evalset\raw\32_evalset_batch4_generated_B.csv


,sha256
perfume_evalset_batch1_A.csv,bea0fced5b608ba4
perfume_evalset_batch2_A.csv,e57085c924a0f260
perfume_evalset_batch3_A.csv,c9217cdf128ddc34
perfume_evalset_batch4_A.csv,4dc7e570be03a64b
32_evalset_batch1_generated_B.csv,ad75503c897876da
32_evalset_batch2_generated_B.csv,9f43b53d70f6fa7f
32_evalset_batch3_generated_B.csv,c58d716fcf8292ed
32_evalset_batch4_generated_B.csv,5c42f7d5483dc7d3


## 3. 읽어서 한 갈래당 한 표로 정규화

In [4]:
frames = {}
for arm, paths in collected.items():
    parts = []
    for p in paths:
        d = pd.read_csv(p)
        d.columns = [c.strip().strip('"') for c in d.columns]
        m = re.search(r"batch(\d+)", p.name)
        d["batch"] = int(m.group(1)) if m else pd.NA
        d["src_file"] = p.name
        parts.append(d[["perfume_id", "sentence_no", "sentence", "batch", "src_file"]])
    frames[arm] = pd.concat(parts, ignore_index=True).sort_values(
        ["perfume_id", "sentence_no"]).reset_index(drop=True)
    print(f"갈래 {arm}  {len(frames[arm])}행 / 향수 {frames[arm]['perfume_id'].nunique()}개")

display(frames["A"].head(3))

갈래 A  300행 / 향수 100개
갈래 B  300행 / 향수 100개


,perfume_id,sentence_no,sentence,batch,src_file
0,6,1,하얀 꽃 향이 진하면서 깨끗한 비누처럼 밝은 느낌이 있었으면 좋겠어. 나무 향과 보송한 분냄새도 은근히 원해.,1,perfume_evalset_batch1_A.csv
1,6,2,단정하고 클래식한 향을 찾고 있어. 꽃 향과 깨끗한 세탁물 같은 느낌이 나고 포근한 분 향이 남았으면 좋겠어.,1,perfume_evalset_batch1_A.csv
2,6,3,꽃다발 같은 향에 숲의 나무 냄새가 섞였으면 좋겠어. 상쾌하면서도 화장품 파우더처럼 보송한 느낌도 있으면 해.,1,perfume_evalset_batch1_A.csv


## 4. 검증 — 사전 등록 §6 규칙 대조

재생성 사유가 되는 것은 **형식 오류와 규칙 2·3 위반뿐**이다.
길이나 분포가 기대와 다른 것은 사유가 아니다 — 고치면 사전 등록이 깨진다.

In [5]:
key = pd.read_csv(INPUT_PATHS["answer_key"])
expected = set(key["perfume_id"])

meta = pd.read_csv(INPUT_PATHS["perfumes_csv"], usecols=["id", "name", "brand"],
                   low_memory=False)
meta = meta[meta["id"].isin(expected)]
name_map = {r["id"]: f"{r['name']} {r['brand']}" for r in meta.to_dict("records")}
ENG = re.compile(r"[A-Za-z]{3,}")

checks = []
for arm, d in frames.items():
    got = set(d["perfume_id"])
    per = d.groupby("perfume_id").size()
    text = d["sentence"].astype(str)

    name_hits = 0
    for r in d.to_dict("records"):
        full = name_map.get(r["perfume_id"], "")
        s = str(r["sentence"])
        for tok in re.split(r"[^A-Za-z가-힣]+", full):
            if len(tok) >= 3 and tok in s:
                name_hits += 1

    dup_in = int(d.groupby("perfume_id")["sentence"]
                 .apply(lambda s: len(s) - s.nunique()).sum())
    checks += [
        {"갈래": arm, "검사": "표본 누락", "값": len(expected - got), "재생성 사유": "예"},
        {"갈래": arm, "검사": "표본 여분", "값": len(got - expected), "재생성 사유": "예"},
        {"갈래": arm, "검사": f"문장 {N_SENTENCES}개가 아닌 향수",
         "값": int((per != N_SENTENCES).sum()), "재생성 사유": "예"},
        {"갈래": arm, "검사": "규칙2 위반 — 영어 향 용어",
         "값": int(text.str.contains(ENG).sum()), "재생성 사유": "예"},
        {"갈래": arm, "검사": "규칙3 위반 — 향수 이름·브랜드",
         "값": name_hits, "재생성 사유": "예"},
        {"갈래": arm, "검사": "같은 향수 안 문장 중복", "값": dup_in, "재생성 사유": "예"},
        {"갈래": arm, "검사": f"{MAX_LEN}자 초과",
         "값": int((text.str.len() > MAX_LEN).sum()), "재생성 사유": "아니오"},
    ]

validation = pd.DataFrame(checks)
display(validation.pivot(index="검사", columns="갈래", values="값"))

blocking = validation[(validation["재생성 사유"] == "예") & (validation["값"] > 0)]
print("검증 통과 — 재생성 사유 없음" if blocking.empty
      else f"검증 실패 — 재생성 사유 {len(blocking)}건")
display(blocking)

갈래,A,B
검사,,
100자 초과,0,0
같은 향수 안 문장 중복,0,0
규칙2 위반 — 영어 향 용어,0,0
규칙3 위반 — 향수 이름·브랜드,0,0
문장 3개가 아닌 향수,0,0
표본 누락,0,0
표본 여분,0,0


검증 통과 — 재생성 사유 없음


,갈래,검사,값,재생성 사유


## 5. 분포 비교 — 이 평가셋이 실사용과 얼마나 닮았나

`spec.md` §7.2 가 golden 200 과 설문 155 에 대해 기록한 값을 **먼저 재현**한다.
재현되지 않으면 아래 숫자를 믿을 수 없다.

In [6]:
TERMS = ["잔향", "지속력", "노트", "베이스", "탑", "미들", "발향", "리니어", "부향률", "시향"]
TERM_PAT = re.compile("|".join(TERMS))


def profile(texts, label):
    """문장 묶음의 길이·전문용어·절 개수. dict."""
    s = pd.Series([str(t) for t in texts if str(t).strip() and str(t) != "nan"])
    clauses = s.map(lambda t: len([c for c in re.split(r"[.!?,]\s*", t) if c.strip()]))
    return {"집단": label, "건수": len(s),
            "평균 길이": round(s.str.len().mean(), 1),
            "길이 중앙값": int(s.str.len().median()),
            "전문 용어 비율": round(s.str.contains(TERM_PAT).mean(), 3),
            "절 개수(대리)": round(clauses.mean(), 2)}


survey = pd.read_csv(INPUT_PATHS["survey"])
golden = pd.read_excel(INPUT_PATHS["golden"])
gcol = next((c for c in golden.columns
             if golden[c].astype(str).str.len().mean() > 15), None)

rows = [profile(golden[gcol], "golden 200 (팀원 작성)"),
        profile(frames["A"]["sentence"], "생성 갈래 A (accord)"),
        profile(frames["B"]["sentence"], "생성 갈래 B (ai_summary)"),
        profile(survey["query_text"], "설문 155 (실사용)")]
distribution = pd.DataFrame(rows)
display(distribution)

# 재현 게이트 — spec §7.2 기록값
gate = []
for label, col, expect, tol in [
        ("golden 200 평균 길이", "평균 길이", 30, 1.0),
        ("golden 200 전문 용어", "전문 용어 비율", 0.055, 0.005),
        ("설문 155 평균 길이", "평균 길이", 71, 1.0),
        ("설문 155 전문 용어", "전문 용어 비율", 0.181, 0.005)]:
    src = "golden" if "golden" in label else "설문"
    got = distribution.loc[distribution["집단"].str.contains(
        "golden" if src == "golden" else "설문"), col].iloc[0]
    gate.append({"항목": label, "spec §7.2": expect, "재측정": got,
                 "일치": abs(got - expect) <= tol})
gate_df = pd.DataFrame(gate)
display(gate_df)
if not gate_df["일치"].all():
    raise RuntimeError("재현 게이트 실패: spec §7.2 의 분포 수치를 재현하지 못했다")
print("재현 게이트 통과 — spec §7.2 의 4개 수치 일치")

,집단,건수,평균 길이,길이 중앙값,전문 용어 비율,절 개수(대리)
0,golden 200 (팀원 작성),201,29.6,29,0.055,1.25
1,생성 갈래 A (accord),300,56.7,57,0.000,2.00
2,생성 갈래 B (ai_summary),300,58.4,58,0.000,2.02
3,설문 155 (실사용),155,70.9,57,0.181,2.72


,항목,spec §7.2,재측정,일치
0,golden 200 평균 길이,30.000,29.600,True
1,golden 200 전문 용어,0.055,0.055,True
2,설문 155 평균 길이,71.000,70.900,True
3,설문 155 전문 용어,0.181,0.181,True


재현 게이트 통과 — spec §7.2 의 4개 수치 일치


### 무엇이 닮았고 무엇이 다른가

길이는 잡혔고 어휘는 못 잡혔다. **결과를 보기 전에 기록한다.**

In [7]:
a = distribution.set_index("집단")
gen = a.loc["생성 갈래 B (ai_summary)"]
sur = a.loc["설문 155 (실사용)"]
gol = a.loc["golden 200 (팀원 작성)"]

print(f"길이 중앙값     golden {gol['길이 중앙값']:.0f}자 → 생성 {gen['길이 중앙값']:.0f}자"
      f" / 설문 {sur['길이 중앙값']:.0f}자")
print(f"절 개수        golden {gol['절 개수(대리)']:.2f} → 생성 {gen['절 개수(대리)']:.2f}"
      f" / 설문 {sur['절 개수(대리)']:.2f}")
print(f"전문 용어      golden {gol['전문 용어 비율']:.1%} → 생성 {gen['전문 용어 비율']:.1%}"
      f" / 설문 {sur['전문 용어 비율']:.1%}   ← 메우지 못했다")
print()
print("생성 문장에 없는 것 — 설문에는 있다")
# '어울리' 는 '겨울에 잘 어울리는' 같은 상황 조건도 잡으므로 쓰지 않는다.
# 1인칭 지시어만 개인화로 센다.
for pat, name in [(r"가격|비싸|저렴|만원|가성비|가격대", "가격 조건"),
                  (r"두통|민감|알레르기|임신|비염", "건강 조건"),
                  (r"나에게|나한테|저에게|저한테|내 스타일|제 스타일|나랑|저랑"
                   r"|나에겐|내게|나와", "개인화 요청")]:
    p = re.compile(pat)
    g = pd.concat([frames["A"]["sentence"], frames["B"]["sentence"]]).astype(str)
    print(f"  {name:8s}  생성 600건 중 {g.str.contains(p).sum():>3}건"
          f"   /   설문 155건 중 {survey['query_text'].astype(str).str.contains(p).sum():>3}건")

길이 중앙값     golden 29자 → 생성 58자 / 설문 57자
절 개수        golden 1.25 → 생성 2.02 / 설문 2.72
전문 용어      golden 5.5% → 생성 0.0% / 설문 18.1%   ← 메우지 못했다

생성 문장에 없는 것 — 설문에는 있다
  가격 조건     생성 600건 중   2건   /   설문 155건 중  25건
  건강 조건     생성 600건 중   0건   /   설문 155건 중   1건
  개인화 요청    생성 600건 중   0건   /   설문 155건 중  10건


## 6. 저장과 해시 고정

In [8]:
checksums = {
    "raw": raw_hashes,
    "answer_key": input_hashes_before["answer_key"],
    "manifest": input_hashes_before["manifest"],
    "note": ("raw/ 는 사람이 GPT 로 만든 원본이다. 이후 바뀌면 이 해시로 감지한다. "
             "사전 등록 §10 — 생성 결과는 받은 파일을 그대로 고정한다."),
}

write_output(OUTPUT_PATHS["generated_a"],
             lambda p: frames["A"].to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["generated_b"],
             lambda p: frames["B"].to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["validation"],
             lambda p: validation.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["distribution"],
             lambda p: distribution.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["checksums"],
             lambda p: p.write_text(json.dumps(checksums, ensure_ascii=False, indent=2),
                                    encoding="utf-8"))

저장: evaluation_data\nlr_evalset\generated_A.csv
저장: evaluation_data\nlr_evalset\generated_B.csv
저장: analysis_outputs\33_evalset_validation.csv
저장: analysis_outputs\33_evalset_distribution.csv
저장: evaluation_data\nlr_evalset\checksums.json


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/evaluation_data/nlr_evalset/checksums.json')

## 7. 가드 검증 — 보호 파일이 안 바뀌었는지

In [9]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("다음 — 노트북 34 에서 기준선을 측정한다.")
print("  현재 설계(하드 AND)와 노트북 11(소프트 점수)로 각각 채점하고")
print("  Condition Precision@5 · 원본 Recall@5 · 갈래 A−B 차이를 낸다.")

입력 해시 불변 확인: answer_key, manifest, survey, golden, perfumes_csv

다음 — 노트북 34 에서 기준선을 측정한다.
  현재 설계(하드 AND)와 노트북 11(소프트 점수)로 각각 채점하고
  Condition Precision@5 · 원본 Recall@5 · 갈래 A−B 차이를 낸다.
